# L4 · Donner des outils à un agent 🛠️

⏱️ **Durée : 35 à 40 minutes** · Niveau : débutant · Modèle : Mistral via LangChain

Dans ce notebook, nous allons transformer une simple fonction Python en **outil utilisable par un agent**. Notre fil rouge reste volontairement familier : une calculatrice.

### 🎯 À la fin, vous saurez

- transformer une fonction Python en tool LangChain avec `@tool` ;
- expliquer comment sa description et son schéma guident Mistral ;
- lire une trace complète : demande d'outil, exécution Python, `ToolMessage`, réponse finale ;
- vérifier qu'un calcul vient réellement de l'outil et pas seulement du modèle.

> **Prérequis :** savoir exécuter des cellules Python et reconnaître une fonction. Aucune connaissance préalable des agents n'est nécessaire.

## 🧠 Pourquoi donner un outil au modèle ?

Un modèle de langage est excellent pour comprendre une demande et formuler une réponse. Mais il n'est pas, à lui seul, une calculatrice, une base de données ou un service météo.

Imaginez un **chef d'atelier** :

- **Mistral** comprend la commande et choisit la bonne machine ;
- **LangChain** transmet la fiche de travail et coordonne les étapes ;
- **Python** fait réellement tourner la machine ;
- le résultat revient au chef, qui l'explique à l'utilisateur.

Sans tool, le modèle peut estimer ou calculer de tête. Avec un tool, notre application lui donne une capacité explicite, observable et testable.

## 📖 Mini-glossaire

| Terme | Définition simple |
|---|---|
| **Tool** | Fonction décrite de manière à pouvoir être choisie par le modèle. |
| **Schéma d'arguments** | Contrat qui précise les noms et types des entrées attendues. |
| **Tool call** | Demande structurée produite par le modèle pour appeler un tool. |
| **`ToolMessage`** | Message contenant le résultat renvoyé par le code après l'exécution du tool. |
| **Agent** | Boucle qui relie le modèle, les tools et les messages jusqu'à une réponse finale. |

## 🗺️ Le trajet d'un calcul

```text
Question de l'utilisateur
        ↓
Mistral choisit un tool et prépare ses arguments
        ↓
LangChain reçoit le tool call
        ↓
Python exécute la fonction
        ↓
LangChain crée un ToolMessage avec le résultat
        ↓
Mistral formule la réponse finale
```

⚠️ **Point essentiel :** Mistral demande l'action, mais c'est bien notre application Python qui l'exécute. La documentation Mistral décrit ce cycle en cinq étapes dans son guide officiel du [function calling](https://docs.mistral.ai/studio/conversations/function-calling).

## 🛠️ Préparer le modèle

LangChain fournit l'intégration officielle [`ChatMistralAI`](https://docs.langchain.com/oss/python/integrations/chat/mistralai). Elle permet d'utiliser un modèle Mistral dans les mêmes abstractions que les autres modèles compatibles LangChain.

Ce notebook lit uniquement les variables déjà présentes dans le **processus courant** :

- `MISTRAL_API_KEY` ;
- `MISTRAL_SERVER_URL`.

Il ne charge, n'affiche et ne modifie aucun fichier `.env`. La clé et l'URL ne sont jamais imprimées.

In [ ]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
import os

from langchain_mistralai import ChatMistralAI

MODEL = "mistral-medium-latest"

# 📚 https://docs.langchain.com/oss/python/integrations/chat/mistralai
mistral_model = ChatMistralAI(
    model=MODEL,
    temperature=0,
    endpoint=os.environ["MISTRAL_SERVER_URL"] + "/v1",
)

print(f"✅ Modèle configuré : {MODEL} · temperature=0")

> 👀 **Résultat attendu**
>
> La cellule affiche uniquement le nom du modèle et `temperature=0`. Si une variable manque, elle s'arrête avec un message explicite sans révéler de secret.

> ⚠️ `temperature=0` réduit la variabilité des réponses, mais ne transforme pas un modèle génératif en programme parfaitement déterministe. Nous vérifierons donc la trace plutôt que de supposer qu'un tool a été appelé.

## 1 · Partir d'une fonction Python ordinaire

Avant LangChain, construisons la « machine » elle-même. Cette fonction reçoit deux nombres et une opération, puis renvoie un nombre. Elle ne connaît ni Mistral, ni les messages, ni les agents.

> 🧠 **Pause prédiction**
>
> Avant d'exécuter la cellule, calculez mentalement la forme du résultat : recevra-t-on du texte, un dictionnaire ou un nombre Python ?

In [2]:
from typing import Literal

Operation = Literal["add", "subtract", "multiply", "divide"]


def calculer_nombres_reels(a: float, b: float, operation: Operation) -> float:
    """Effectue une opération arithmétique en Python pur."""
    if operation == "add":
        return a + b
    if operation == "subtract":
        return a - b
    if operation == "multiply":
        return a * b
    if operation == "divide":
        if b == 0:
            raise ValueError("La division par zéro est interdite.")
        return a / b
    raise ValueError(f"Opération inconnue : {operation}")


resultat_python = calculer_nombres_reels(3.1125, 4.1234, "multiply")
print(resultat_python, type(resultat_python))

12.8340825 <class 'float'>


> 👀 **Résultat attendu :** `12.8340825 <class 'float'>`.

> 🔍 **Lecture de la sortie**
>
> Python a exécuté directement la fonction. Aucun modèle n'a choisi l'opération et aucun message LangChain n'a circulé. Nous avons la capacité, mais pas encore la fiche qui permet à Mistral de la découvrir.

## 2 · Transformer la fonction en tool

Le décorateur [`@tool`](https://docs.langchain.com/oss/python/langchain/tools) transforme une fonction en objet LangChain. À partir du nom, des annotations de types et de la docstring, LangChain construit une description et un schéma transmis au modèle.

> 🧠 **Pause prédiction**
>
> Avec `a: float`, `b: float` et quatre opérations possibles, quelles contraintes devraient apparaître dans le schéma ?

In [3]:
from langchain.tools import tool

# 📚 Doc officielle LangChain :
# https://docs.langchain.com/oss/python/langchain/tools
@tool
def real_number_calculator(a: float, b: float, operation: Operation) -> float:
    """Effectue une opération arithmétique sur deux nombres réels."""
    print("🧮 Python exécute real_number_calculator")
    return calculer_nombres_reels(a, b, operation)

In [4]:
from pprint import pprint

print("Nom :", real_number_calculator.name)
print("Description :", real_number_calculator.description)
print("Schéma envoyé au modèle :")
pprint(real_number_calculator.args_schema.model_json_schema())

Nom : real_number_calculator
Description : Effectue une opération arithmétique sur deux nombres réels.
Schéma envoyé au modèle :
{'description': 'Effectue une opération arithmétique sur deux nombres réels.',
 'properties': {'a': {'title': 'A', 'type': 'number'},
                'b': {'title': 'B', 'type': 'number'},
                'operation': {'enum': ['add', 'subtract', 'multiply', 'divide'],
                              'title': 'Operation',
                              'type': 'string'}},
 'required': ['a', 'b', 'operation'],
 'title': 'real_number_calculator',
 'type': 'object'}


> 👀 **Résultat attendu**
>
> Le schéma contient deux nombres, `a` et `b`, puis `operation` limitée à `add`, `subtract`, `multiply` ou `divide`.

> 🔍 **Qui fait quoi ?**
>
> | Acteur | Responsabilité à cette étape |
> |---|---|
> | Python | fournit la fonction, ses types et sa docstring ; |
> | LangChain | fabrique le nom, la description et le schéma ; |
> | Mistral | recevra ce contrat pour décider si et comment appeler le tool. |

## 3 · Confier le tool à un agent

La fonction [`create_agent`](https://docs.langchain.com/oss/python/langchain/agents) construit la boucle modèle ↔ tools. Lui fournir un tool ne l'exécute pas immédiatement : l'agent attend une question, puis Mistral décide s'il souhaite le demander.

In [5]:
from langchain.agents import create_agent

# 📚 Doc officielle LangChain :
# https://docs.langchain.com/oss/python/langchain/agents
agent = create_agent(
    model=mistral_model,
    tools=[real_number_calculator],
    system_prompt=(
        "Tu es un assistant pédagogique. Utilise la calculatrice lorsqu'un calcul "
        "précis est demandé, puis explique brièvement le résultat en français."
    ),
)

print("✅ Agent prêt avec 1 tool :", real_number_calculator.name)

✅ Agent prêt avec 1 tool : real_number_calculator


### ▶️ Poser une question et lire toute la trace

> 🧠 **Pause prédiction**
>
> La réponse finale sera le dernier message. Mais combien d'étapes intermédiaires faut-il pour que Python calcule réellement `3.1125 × 4.1234` ? Repérez-les dans le schéma mental du début.

In [6]:
from langchain_core.messages import AIMessage, ToolMessage


def afficher_trace(messages: list) -> None:
    """Affiche les décisions du modèle et les résultats d'outils sans les masquer."""
    for numero, message in enumerate(messages, start=1):
        print(f"\n--- Étape {numero} · {type(message).__name__} ---")
        if isinstance(message, AIMessage) and message.tool_calls:
            for appel in message.tool_calls:
                print("🔧 Tool call :", appel["name"])
                print("   Arguments :", appel["args"])
        elif isinstance(message, ToolMessage):
            print("📦 Résultat Python :", message.content)
        else:
            print(message.content)


# 📚 Cycle officiel Mistral du function calling :
# https://docs.mistral.ai/studio/conversations/function-calling
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Combien font 3.1125 multiplié par 4.1234 ?",
            }
        ]
    }
)

afficher_trace(result["messages"])

🧮 Python exécute real_number_calculator

--- Étape 1 · HumanMessage ---
Combien font 3.1125 multiplié par 4.1234 ?

--- Étape 2 · AIMessage ---
🔧 Tool call : real_number_calculator
   Arguments : {'a': 3.1125, 'b': 4.1234, 'operation': 'multiply'}

--- Étape 3 · ToolMessage ---
📦 Résultat Python : 12.8340825

--- Étape 4 · AIMessage ---
Le résultat de **3.1125 multiplié par 4.1234** est **12.8340825**.


> 👀 **Résultat attendu — la forme compte plus que la formulation**
>
> 1. un message utilisateur ;
> 2. un `AIMessage` avec un tool call nommé `real_number_calculator` et l'opération `multiply` ;
> 3. la ligne `🧮 Python exécute...`, preuve que notre code tourne ;
> 4. un `ToolMessage` contenant `12.8340825` ;
> 5. un dernier `AIMessage` qui formule la réponse.

> 🔍 **Ne confondez pas les deux `AIMessage`**
>
> Le premier ne donne pas encore la réponse : il demande une action structurée. Le second arrive après le résultat Python et peut enfin répondre à l'utilisateur. Si la formulation finale varie légèrement, vérifiez les `tool_calls` et le `ToolMessage` : ce sont les preuves stables.

## 4 · Pourquoi la description du tool compte

Le nom et le schéma disent **comment** appeler l'outil. La description explique surtout **quand** l'utiliser. C'est l'équivalent de l'étiquette collée sur une machine.

LangChain peut analyser une docstring au format Google avec `parse_docstring=True`. Nous créons une seconde version, plus explicite, sans écraser la première.

> 🧠 **Pause prédiction**
>
> Pour `3 × 4`, le modèle pourrait calculer mentalement. Quelle phrase de la description lui indique malgré tout d'utiliser notre tool ?

In [7]:
@tool(
    "calculator",
    parse_docstring=True,
    description=(
        "Effectue des opérations arithmétiques fiables. "
        "Utilise toujours cet outil pour un calcul demandé, même avec des entiers."
    ),
)
def calculatrice_documentee(a: float, b: float, operation: Operation) -> float:
    """Calcule précisément une opération sur deux nombres.

    Args:
        a: Premier nombre du calcul.
        b: Second nombre du calcul.
        operation: Opération à effectuer entre les deux nombres.

    Returns:
        Le résultat numérique calculé par Python.
    """
    print("🧮 Python exécute calculator")
    return calculer_nombres_reels(a, b, operation)


print("Description enrichie :", calculatrice_documentee.description)
pprint(calculatrice_documentee.args_schema.model_json_schema())

Description enrichie : Effectue des opérations arithmétiques fiables. Utilise toujours cet outil pour un calcul demandé, même avec des entiers.
{'description': 'Calcule précisément une opération sur deux nombres.',
 'properties': {'a': {'description': 'Premier nombre du calcul.',
                      'title': 'A',
                      'type': 'number'},
                'b': {'description': 'Second nombre du calcul.',
                      'title': 'B',
                      'type': 'number'},
                'operation': {'description': 'Opération à effectuer entre les '
                                             'deux nombres.',
                              'enum': ['add', 'subtract', 'multiply', 'divide'],
                              'title': 'Operation',
                              'type': 'string'}},
 'required': ['a', 'b', 'operation'],
 'title': 'calculator',
 'type': 'object'}


In [8]:
agent_documente = create_agent(
    model=mistral_model,
    tools=[calculatrice_documentee],
    system_prompt="Réponds en français et appuie-toi sur les tools disponibles.",
)

resultat_entiers = agent_documente.invoke(
    {"messages": [{"role": "user", "content": "Combien font 3 × 4 ?"}]}
)
afficher_trace(resultat_entiers["messages"])

🧮 Python exécute calculator

--- Étape 1 · HumanMessage ---
Combien font 3 × 4 ?

--- Étape 2 · AIMessage ---
🔧 Tool call : calculator
   Arguments : {'a': 3, 'b': 4, 'operation': 'multiply'}

--- Étape 3 · ToolMessage ---
📦 Résultat Python : 12.0

--- Étape 4 · AIMessage ---
3 × 4 font **12**.


> 👀 **Résultat attendu**
>
> La trace doit montrer un appel à `calculator` avec `a=3`, `b=4` et `operation="multiply"`, puis un `ToolMessage` contenant `12`.

> ⚠️ **Limite honnête**
>
> Une meilleure description augmente la probabilité d'un bon choix ; elle ne constitue pas une règle de sécurité. Selon le modèle et l'endpoint, un comportement génératif peut encore varier. Pour une obligation métier, ajoutez aussi des contrôles déterministes dans votre application.

## 🧪 Micro-exercice · Calculer un prix TTC

Votre collègue souhaite un tool qui calcule un prix toutes taxes comprises :

```text
prix TTC = prix HT × (1 + taux de TVA / 100)
```

Complétez le squelette ci-dessous, puis demandez : **« Quel est le prix TTC de 80 € avec 20 % de TVA ? »**

### ✅ Critères de réussite

- le schéma expose `prix_ht` et `taux_tva` comme nombres ;
- la trace contient un tool call nommé `calculer_prix_ttc` ;
- le `ToolMessage` et la réponse finale donnent **96 €**.

💡 **Indice :** reprenez le décorateur, la docstring et `afficher_trace` utilisés plus haut.

In [9]:
# 👉 À vous : décommentez puis remplacez les TODO.
# Le squelette reste commenté pour que « Run All » fonctionne avant l'exercice.
#
# @tool("calculer_prix_ttc", parse_docstring=True)
# def calculer_prix_ttc(prix_ht: float, taux_tva: float) -> float:
#     """Calcule un prix TTC à partir d'un prix HT et d'un taux de TVA.
#
#     Args:
#         prix_ht: TODO
#         taux_tva: TODO
#
#     Returns:
#         TODO
#     """
#     return TODO
#
# agent_ttc = create_agent(model=mistral_model, tools=[calculer_prix_ttc])
# resultat_ttc = agent_ttc.invoke(
#     {"messages": [{"role": "user", "content": "Quel est le prix TTC de 80 € avec 20 % de TVA ?"}]}
# )
# afficher_trace(resultat_ttc["messages"])

<details>
<summary>✅ Voir une correction complète</summary>

```python
@tool("calculer_prix_ttc", parse_docstring=True)
def calculer_prix_ttc(prix_ht: float, taux_tva: float) -> float:
    """Calcule un prix TTC à partir d'un prix HT et d'un taux de TVA.

    Args:
        prix_ht: Prix hors taxes, exprimé en euros.
        taux_tva: Pourcentage de TVA à appliquer, par exemple 20 pour 20 %.

    Returns:
        Le prix TTC arrondi à deux décimales.
    """
    return round(prix_ht * (1 + taux_tva / 100), 2)

agent_ttc = create_agent(
    model=mistral_model,
    tools=[calculer_prix_ttc],
    system_prompt="Utilise le tool pour tout calcul de prix TTC et réponds en français.",
)
resultat_ttc = agent_ttc.invoke(
    {"messages": [{"role": "user", "content": "Quel est le prix TTC de 80 € avec 20 % de TVA ?"}]}
)
afficher_trace(resultat_ttc["messages"])
```

La valeur Python attendue est `96.0`. La formulation du dernier message peut varier, mais le tool call et le `ToolMessage` doivent être visibles.

</details>

## 🧭 Ce qu'il faut retenir

Vous savez maintenant :

- ✅ séparer la logique Python de sa présentation comme tool ;
- ✅ lire le nom, la description et le schéma transmis au modèle ;
- ✅ distinguer la décision de Mistral de l'exécution réelle par Python ;
- ✅ repérer un tool call et le `ToolMessage` correspondant ;
- ✅ vérifier une trace au lieu de vous fier uniquement à la réponse finale.

### ⚠️ Du notebook à la production

| Dans cette démonstration | En production |
|---|---|
| calcul local sans effet de bord | validation stricte des arguments et gestion des erreurs |
| `print()` pour observer | traces structurées et supervision |
| description comme consigne | contrôles d'autorisation dans le code |
| un seul tool | sélection, permissions et tests pour chaque tool |

### 🧭 Transition vers L5

Notre calculatrice est une fonction locale connue à l'avance. Dans **L5**, nous brancherons un serveur **MCP** : l'agent découvrira des tools externes grâce à un protocole standard, comme un ordinateur reconnaît un périphérique connecté.

## 📚 Documentation officielle

- [LangChain · Tools](https://docs.langchain.com/oss/python/langchain/tools) — créer un tool, documenter ses arguments et inspecter son schéma.
- [LangChain · Agents](https://docs.langchain.com/oss/python/langchain/agents) — comprendre la boucle `create_agent`, les messages et les appels d'outils.
- [LangChain · Intégration ChatMistralAI](https://docs.langchain.com/oss/python/integrations/chat/mistralai) — configurer les modèles Mistral et leurs capacités dans LangChain.
- [Mistral · Function calling](https://docs.mistral.ai/studio/conversations/function-calling) — revoir les cinq étapes entre la demande du modèle et le retour du résultat d'outil.